# 08 — EDA: Joined Dataset (flights_featured)

With all data sources merged into one flight-level table we can ask questions that required
multiple notebooks before: how does weather intensity scale with delay, which departure windows
are most exposed to storms, do monopoly routes behave differently, and which cities feed the most
passengers into DMV airports.

**Source:** `data/processed/flights_featured.parquet` — 2.8 M rows, all DMV departures 2015-2025.

---
| Section | Topic |
|---------|-------|
| 1 | Dataset Overview |
| 2 | Visibility Threshold Effect on Delay |
| 3 | Weather Type vs. Delay Severity |
| 4 | Departure Window × Thunderstorm Interaction |
| 5 | Season × Airport Delay Heatmap |
| 6 | Carrier On-Time Performance by Airport |
| 7 | Route Monopoly and On-Time Rate |
| 8 | Fare Level vs. Delay |
| 9 | Inbound Feeder Cities into DMV |
| 10 | Summary & Key Findings |

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import PROCESSED_DATA_PATH

AIRPORTS = ['IAD', 'DCA', 'BWI']
AIRPORT_COLORS = {'IAD': '#1f77b4', 'DCA': '#ff7f0e', 'BWI': '#2ca02c'}

feat = pd.read_parquet(PROJECT_ROOT / PROCESSED_DATA_PATH / 'flights_featured.parquet')
feat['FlightDate'] = pd.to_datetime(feat['FlightDate'])

# Operated (non-cancelled) flights are the basis for delay analysis
operated = feat[feat['Cancelled'] == 0].copy()

print(f'flights_featured: {len(feat):,} total rows  ({len(operated):,} operated)')
print(f'Columns: {feat.shape[1]}')
print(f'Date range: {feat["FlightDate"].min().date()} to {feat["FlightDate"].max().date()}')

## Section 1 — Dataset Overview

Quick snapshot of the joined dataset: join coverage, target variable distributions,
and the overall share of feature columns that are populated.

In [ ]:
print('=== Join Coverage (all rows) ===')
for label, col in [
    ('NOAA weather',   'wx_min_vis'),
    ('T-100 capacity', 't100_seats'),
    ('DB1B fare',      'db1b_avg_fare'),
]:
    n = int(feat[col].notna().sum())
    pct = n / len(feat) * 100
    print(f'  {label:<18}: {n:>10,}  ({pct:.1f}%)')

print()
print('=== Delay & Cancellation Summary ===')
cancel_pct  = feat['Cancelled'].mean() * 100
late_pct    = operated['is_late'].mean() * 100
mean_delay  = operated['ArrDelayMinutes'].mean()
print(f'  Cancellation rate:          {cancel_pct:.2f}%')
print(f'  Late rate (operated):       {late_pct:.1f}%')
print(f'  Mean arrival delay:         {mean_delay:.1f} min')

print()
print('=== Late Rate by Airport ===')
for ap in AIRPORTS:
    sub = operated[operated['Origin'] == ap]
    lr  = sub['is_late'].mean() * 100
    md  = sub['ArrDelayMinutes'].mean()
    print(f'  {ap}: {lr:.1f}% late  |  {md:.1f} min mean delay  |  {len(sub):,} flights')

## Section 2 — Visibility Threshold Effect on Delay

Aviation visibility is categorised into FAA flight-rule bands. If low visibility directly drives
delays we should see a step-change at the IFR threshold (3 mi) and again at LIFR (1 mi).
Using the minimum daily visibility recorded at the departure airport, we compare mean delay
and late rate across visibility bands.

In [ ]:
vis = operated['wx_min_vis']
operated['vis_cat'] = np.select(
    [vis.isna(), vis < 1, vis < 3, vis < 5, vis < 10],
    ['No wx data', 'LIFR (<1 mi)', 'IFR (1-3 mi)', 'MVFR (3-5 mi)', 'VFR (5-10 mi)'],
    default='Clear (>10 mi)',
)

vis_order = ['LIFR (<1 mi)', 'IFR (1-3 mi)', 'MVFR (3-5 mi)', 'VFR (5-10 mi)', 'Clear (>10 mi)', 'No wx data']
vis_stats = (
    operated.groupby('vis_cat')
    .agg(
        mean_delay = ('ArrDelayMinutes', 'mean'),
        late_pct   = ('is_late',         'mean'),
        n_flights  = ('ArrDelayMinutes', 'count'),
    )
    .reindex(vis_order)
    .round(3)
)
vis_stats['late_pct_disp'] = (vis_stats['late_pct'] * 100).round(1)
print(vis_stats[['mean_delay', 'late_pct_disp', 'n_flights']].to_string())
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cats = [c for c in vis_order if c in vis_stats.index and vis_stats.loc[c, 'n_flights'] > 100]
colors = ['#d62728','#ff7f0e','#ffbb78','#aec7e8','#1f77b4','#c7c7c7']

ax = axes[0]
vals = [vis_stats.loc[c, 'mean_delay'] for c in cats]
ax.bar(range(len(cats)), vals, color=colors[:len(cats)], alpha=0.85)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Mean Arrival Delay (min)')
ax.set_title('Mean Delay by Visibility Band')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
vals = [vis_stats.loc[c, 'late_pct_disp'] for c in cats]
ax.bar(range(len(cats)), vals, color=colors[:len(cats)], alpha=0.85)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('% Flights Late (> 15 min)')
ax.set_title('Late Rate by Visibility Band')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Visibility Band vs. Delay  (all DMV departures, 2015-2025)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 3 — Weather Type vs. Delay Severity

Thunderstorms, snow, and fog are operationally distinct: thunderstorms are brief and intense,
snow can last hours and close runways, fog degrades visibility without wind.
We assign each flight-day to the most operationally severe condition observed
(thunderstorm > snow > fog > clear) and compare delay profiles.

In [ ]:
ts   = operated['wx_had_ts'].fillna(False)
snow = operated['wx_had_snow'].fillna(False)
fog  = operated['wx_had_fog'].fillna(False)

operated['wx_type'] = np.select(
    [ts, snow, fog],
    ['Thunderstorm', 'Snow', 'Fog / Mist'],
    default='Clear / Unknown',
)

wx_order = ['Thunderstorm', 'Snow', 'Fog / Mist', 'Clear / Unknown']
wx_stats = (
    operated.groupby('wx_type')
    .agg(
        mean_delay  = ('ArrDelayMinutes', 'mean'),
        late_pct    = ('is_late',         'mean'),
        cancel_pct  = ('Cancelled',       lambda x: 0),
        n_flights   = ('ArrDelayMinutes', 'count'),
    )
    .reindex(wx_order)
    .round(3)
)
wx_stats['late_pct_disp'] = (wx_stats['late_pct'] * 100).round(1)
print(wx_stats[['mean_delay', 'late_pct_disp', 'n_flights']].rename(
    columns={'mean_delay':'Mean Delay (min)','late_pct_disp':'Late %','n_flights':'Flights'}
).to_string())
print()

wx_colors = {'Thunderstorm': '#d62728', 'Snow': '#1f77b4', 'Fog / Mist': '#7f7f7f', 'Clear / Unknown': '#2ca02c'}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
vals = [wx_stats.loc[w, 'mean_delay'] for w in wx_order]
ax.bar(range(len(wx_order)), vals, color=[wx_colors[w] for w in wx_order], alpha=0.85)
ax.set_xticks(range(len(wx_order)))
ax.set_xticklabels(wx_order)
ax.set_ylabel('Mean Arrival Delay (min)')
ax.set_title('Mean Delay by Weather Type')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
vals = [wx_stats.loc[w, 'late_pct_disp'] for w in wx_order]
ax.bar(range(len(wx_order)), vals, color=[wx_colors[w] for w in wx_order], alpha=0.85)
ax.set_xticks(range(len(wx_order)))
ax.set_xticklabels(wx_order)
ax.set_ylabel('% Flights Late')
ax.set_title('Late Rate by Weather Type')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Weather Type vs. Delay  (dominant condition per flight-day, 2015-2025)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 4 — Departure Window × Thunderstorm Interaction

Afternoon convection peaks 14-17 EDT (18-21 UTC) — directly overlapping the busiest
departure bank. We test whether the afternoon window is disproportionately harmed
when a thunderstorm occurs, compared to other departure periods.

In [ ]:
bucket_order = ['Early', 'Morning', 'Afternoon', 'Evening', 'Night']
op2 = operated[operated['dep_hour_bucket'].isin(bucket_order)].copy()
op2['had_ts'] = op2['wx_had_ts'].fillna(False)

interaction = (
    op2.groupby(['dep_hour_bucket', 'had_ts'])
    .agg(mean_delay=('ArrDelayMinutes', 'mean'), n=('ArrDelayMinutes', 'count'))
    .reset_index()
)
pivot = interaction.pivot(index='dep_hour_bucket', columns='had_ts', values='mean_delay')
pivot = pivot.reindex(bucket_order)
pivot.columns = ['No Thunderstorm', 'Thunderstorm Day']
pivot['lift'] = pivot['Thunderstorm Day'] - pivot['No Thunderstorm']

print('Mean arrival delay (min) by departure window and thunderstorm:')
print(pivot.round(1).to_string())
print()

x = np.arange(len(bucket_order))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.bar(x - w/2, pivot['No Thunderstorm'],  w, label='No Thunderstorm',  color='#aec7e8', alpha=0.9)
ax.bar(x + w/2, pivot['Thunderstorm Day'], w, label='Thunderstorm Day', color='#d62728', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bucket_order)
ax.set_ylabel('Mean Arrival Delay (min)')
ax.set_title('Delay by Departure Window and Thunderstorm')
ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
lift_vals = pivot['lift'].values
bar_colors = ['#d62728' if v > 0 else '#2ca02c' for v in lift_vals]
ax.bar(x, lift_vals, color=bar_colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(bucket_order)
ax.set_ylabel('Additional Delay on Storm Day (min)')
ax.set_title('Thunderstorm Delay Premium by Departure Window')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Afternoon Departures Pay the Largest Storm Penalty', fontsize=11)
plt.tight_layout()
plt.show()

## Section 5 — Season × Airport Delay Heatmap

Delay drivers differ by season: winter brings snow and ice at IAD (more inland, colder),
summer brings thunderstorms worst at DCA (urban heat, Potomac moisture).
A heatmap makes the airport × season interaction legible at a glance.

In [ ]:
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
ap_season = (
    operated.groupby(['Origin', 'season'])
    .agg(
        late_pct   = ('is_late',          'mean'),
        mean_delay = ('ArrDelayMinutes',  'mean'),
        cancel_pct = ('Cancelled',        lambda x: 0),
        n_flights  = ('ArrDelayMinutes',  'count'),
    )
    .reset_index()
)

pivot_late   = ap_season.pivot(index='Origin', columns='season', values='late_pct')[season_order] * 100
pivot_delay  = ap_season.pivot(index='Origin', columns='season', values='mean_delay')[season_order]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for ax, pivot, title, fmt in zip(
    axes,
    [pivot_late, pivot_delay],
    ['Late Rate (%)', 'Mean Arrival Delay (min)'],
    ['{:.1f}%', '{:.1f}'],
):
    data = pivot.reindex(AIRPORTS).values.astype(float)
    im = ax.imshow(data, cmap='YlOrRd', aspect='auto',
                   vmin=np.nanmin(data), vmax=np.nanmax(data))
    ax.set_xticks(range(len(season_order)))
    ax.set_xticklabels(season_order)
    ax.set_yticks(range(len(AIRPORTS)))
    ax.set_yticklabels(AIRPORTS)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for i in range(len(AIRPORTS)):
        for j in range(len(season_order)):
            v = data[i, j]
            if not np.isnan(v):
                txt = fmt.format(v)
                ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                        color='white' if v > (np.nanmax(data) * 0.65) else 'black')

plt.suptitle('Delay by Airport x Season  (operated flights, 2015-2025)', fontsize=11)
plt.tight_layout()
plt.show()

best_season = pivot_late.reindex(AIRPORTS).idxmin(axis=1)
worst_season = pivot_late.reindex(AIRPORTS).idxmax(axis=1)
print('Best / worst season per airport:')
for ap in AIRPORTS:
    b = best_season[ap]
    w = worst_season[ap]
    bv = pivot_late.loc[ap, b]
    wv = pivot_late.loc[ap, w]
    print(f'  {ap}:  best {b} ({bv:.1f}%)  |  worst {w} ({wv:.1f}%)')

## Section 6 — Carrier On-Time Performance by Airport

Using the full flight-level dataset (not monthly T-100 aggregates), we compute each
carrier's late rate at each DMV airport. Carriers with fewer than 2,000 operated
flights at an airport are excluded to keep comparisons statistically meaningful.

In [ ]:
carrier_ap = (
    operated.groupby(['Origin', 'Reporting_Airline'])
    .agg(
        late_pct   = ('is_late',          'mean'),
        mean_delay = ('ArrDelayMinutes',  'mean'),
        n_flights  = ('ArrDelayMinutes',  'count'),
    )
    .reset_index()
)
carrier_ap = carrier_ap[carrier_ap['n_flights'] >= 2000]
carrier_ap['late_pct_disp'] = carrier_ap['late_pct'] * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ap in zip(axes, AIRPORTS):
    sub = carrier_ap[carrier_ap['Origin'] == ap].sort_values('late_pct')
    bar_colors = [AIRPORT_COLORS[ap] if r['late_pct'] <= sub['late_pct'].median()
                  else '#d62728'
                  for _, r in sub.iterrows()]
    ax.barh(sub['Reporting_Airline'], sub['late_pct_disp'],
            color=bar_colors, alpha=0.85)
    ax.axvline(sub['late_pct_disp'].median(), color='black',
               linestyle='--', linewidth=1, alpha=0.5, label='median')
    ax.set_title(f'{ap} — Carrier Late Rate')
    ax.set_xlabel('% Flights Late (> 15 min)')
    ax.legend(fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Carrier Late Rate by Airport  (min 2,000 operated flights, 2015-2025)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 7 — Route Monopoly and On-Time Rate

Does a carrier's competitive position on a route correlate with its on-time performance?
`t100_carrier_share` measures the fraction of that route's monthly departures the carrier
holds. We bin routes into four tiers and compare delay rates — watching for whether
monopoly routes show a different profile than contested ones.

In [ ]:
share_data = operated[operated['t100_carrier_share'].notna()].copy()
share_data['share_bin'] = pd.cut(
    share_data['t100_carrier_share'],
    bins=[0, 0.4, 0.7, 0.9, 1.001],
    labels=['Contested (<40%)', 'Competitive (40-70%)', 'Dominant (70-90%)', 'Monopoly (>90%)'],
)

share_stats = (
    share_data.groupby('share_bin', observed=True)
    .agg(
        late_pct   = ('is_late',          'mean'),
        mean_delay = ('ArrDelayMinutes',  'mean'),
        n_flights  = ('ArrDelayMinutes',  'count'),
    )
    .round(3)
)
share_stats['late_pct_disp'] = (share_stats['late_pct'] * 100).round(1)
print(share_stats[['mean_delay', 'late_pct_disp', 'n_flights']].rename(
    columns={'mean_delay':'Mean Delay','late_pct_disp':'Late %','n_flights':'Flights'}
).to_string())
print()

tiers = share_stats.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
tier_colors = ['#2ca02c', '#aec7e8', '#ff7f0e', '#d62728']

ax = axes[0]
ax.bar(range(len(tiers)), share_stats['mean_delay'], color=tier_colors, alpha=0.85)
ax.set_xticks(range(len(tiers)))
ax.set_xticklabels(tiers, rotation=15, ha='right', fontsize=8)
ax.set_ylabel('Mean Arrival Delay (min)')
ax.set_title('Mean Delay by Route Competition Tier')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(range(len(tiers)), share_stats['late_pct_disp'], color=tier_colors, alpha=0.85)
ax.set_xticks(range(len(tiers)))
ax.set_xticklabels(tiers, rotation=15, ha='right', fontsize=8)
ax.set_ylabel('% Flights Late')
ax.set_title('Late Rate by Route Competition Tier')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Route Competition Level vs. On-Time Performance', fontsize=11)
plt.tight_layout()
plt.show()

## Section 8 — Fare Level vs. Delay

DB1B average nonstop fare proxies for route type: high-fare thin markets tend to be
business-driven; low-fare dense markets are leisure-dominated.
We split matched flights into fare quartiles and compare on-time performance —
watching for whether premium routes are better protected.

In [ ]:
fare_data = operated[operated['db1b_avg_fare'].notna()].copy()
fare_data['fare_q'] = pd.qcut(
    fare_data['db1b_avg_fare'], 4,
    labels=['Q1 Low', 'Q2', 'Q3', 'Q4 High'],
    duplicates='drop',
)

fare_stats = (
    fare_data.groupby('fare_q', observed=True)
    .agg(
        avg_fare   = ('db1b_avg_fare',    'mean'),
        late_pct   = ('is_late',          'mean'),
        mean_delay = ('ArrDelayMinutes',  'mean'),
        n_flights  = ('ArrDelayMinutes',  'count'),
    )
    .round(2)
)
fare_stats['late_pct_disp'] = (fare_stats['late_pct'] * 100).round(1)
print(fare_stats[['avg_fare','mean_delay','late_pct_disp','n_flights']].rename(
    columns={'avg_fare':'Avg Fare ($)','mean_delay':'Mean Delay','late_pct_disp':'Late %','n_flights':'Flights'}
).to_string())
print()

qs = fare_stats.index.tolist()
q_colors = ['#2ca02c', '#aec7e8', '#ff7f0e', '#d62728']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.bar(range(len(qs)), fare_stats['mean_delay'], color=q_colors, alpha=0.85)
ax.set_xticks(range(len(qs)))
ax.set_xticklabels(qs)
ax.set_ylabel('Mean Arrival Delay (min)')
ax.set_title('Mean Delay by Fare Quartile')
avg_fares = [f'${fare_stats.loc[q, "avg_fare"]:.0f}' for q in qs]
for i, lbl in enumerate(avg_fares):
    ax.text(i, fare_stats['mean_delay'].iloc[i] + 0.1, lbl, ha='center', fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(range(len(qs)), fare_stats['late_pct_disp'], color=q_colors, alpha=0.85)
ax.set_xticks(range(len(qs)))
ax.set_xticklabels(qs)
ax.set_ylabel('% Flights Late')
ax.set_title('Late Rate by Fare Quartile')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Fare Level vs. On-Time Performance  (nonstop DB1B-matched flights)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 9 — Inbound Feeder Cities into DMV

BTS on-time data only captures departures, so the geographic picture of who flies *into*
DMV airports is invisible there. DB1B destination rows provide this view — ticketed
itineraries ending at IAD, DCA, or BWI, grouped by origin airport.

DMV-to-DMV connections (e.g., IAD→DCA) are excluded to focus on external feeder markets.

In [ ]:
db1b_in = pd.read_parquet(
    PROJECT_ROOT / PROCESSED_DATA_PATH / 'db1b_dmv.parquet',
    columns=['Origin', 'Dest', 'Passengers', 'MktFare'],
)
into_dmv = db1b_in[
    db1b_in['Dest'].isin(AIRPORTS) &
    ~db1b_in['Origin'].isin(AIRPORTS)
].copy()
del db1b_in

origin_pax = (
    into_dmv.groupby(['Origin', 'Dest'])['Passengers']
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ap in zip(axes, AIRPORTS):
    sub = origin_pax[origin_pax['Dest'] == ap].nlargest(10, 'Passengers').copy()
    total = sub['Passengers'].sum()
    sub['pct'] = sub['Passengers'] / total * 100
    sub = sub.sort_values('pct')
    ax.barh(sub['Origin'], sub['pct'], color=AIRPORT_COLORS[ap], alpha=0.85)
    ax.set_title(f'Top Feeder Cities into {ap}')
    ax.set_xlabel('% of Sampled Passengers')
    ax.grid(axis='x', alpha=0.3)
    for i, (_, row) in enumerate(sub.reset_index(drop=True).iterrows()):
        pct_v = row['pct']
        ax.text(pct_v + 0.2, i, f'{pct_v:.1f}%', va='center', fontsize=8)

plt.suptitle('DB1B: Top Origin Airports into DMV  (2015-2025, 10% pax sample, excl. DMV-DMV)', fontsize=11)
plt.tight_layout()
plt.show()

print('Top 5 feeder airports per destination:')
for ap in AIRPORTS:
    top5 = origin_pax[origin_pax['Dest'] == ap].nlargest(5, 'Passengers')['Origin'].tolist()
    print(f'  {ap}: {top5}')

## Section 10 — Summary & Key Findings

In [ ]:
print('=== Key Findings: Joined EDA (flights_featured, 2015-2025) ===')
print()
print('1. Visibility shows a step-change at the IFR threshold.')
if 'vis_stats' in dir():
    lifr_d  = vis_stats.loc['LIFR (<1 mi)',  'mean_delay']
    clear_d = vis_stats.loc['Clear (>10 mi)','mean_delay']
    print(f'   LIFR days average {lifr_d:.1f} min delay vs {clear_d:.1f} min on clear days.')
print()
print('2. Thunderstorm vs. snow vs. fog — distinct delay profiles.')
if 'wx_stats' in dir():
    for wt in ['Thunderstorm', 'Snow', 'Fog / Mist']:
        d = wx_stats.loc[wt, 'mean_delay']
        l = wx_stats.loc[wt, 'late_pct_disp']
        print(f'   {wt:<14}: {d:.1f} min mean delay, {l:.1f}% late')
print()
print('3. Afternoon departures pay the largest thunderstorm penalty.')
if 'pivot' in dir():
    pm = pivot['lift'].idxmax()
    pv = pivot.loc[pm, 'lift']
    print(f'   {pm} window: +{pv:.1f} min extra delay on storm days.')
print()
print('4. Season x airport: each airport has a distinct worst season.')
print()
print('5. Route competition and fare level show relationships with delay')
print('   that merit controlled analysis in the modeling notebook.')
print()
print('Next: 09_modeling.ipynb')